# 5.6 · 决策树分类器 / Decision Tree Classifier

> **课程定位 / Where this fits**
> 4.11 讲过回归树。这里是分类版。决策树用一系列"是/否"问题把空间切成**轴对齐的方块**, 每块投一个类。它**可解释性极强**(可画成流程图)、不需缩放、能处理混合类型特征——更重要的是, 它是**随机森林(5.7)、GBDT(5.8)、XGBoost(5.9)** 的基石。
> Decision trees split space into axis-aligned boxes via yes/no questions — highly interpretable, scaling-free, and the building block for RF and all the boosting methods.

> 💡 **面试相关 / Interview-relevant**
> - "Gini 与熵的区别 / 信息增益" ★★★★★
> - "决策树怎么防过拟合(剪枝/超参)" ★★★★★
> - "为什么树不需要特征缩放" ★★★★
> - "决策树为什么是高方差模型" ★★★★（引出 5.7 bagging）
> - "如何处理连续特征的分裂点" ★★★

---

## 学习目标 / Learning Objectives
1. 树如何用**不纯度(Gini/熵)**贪心选分裂。
2. 从零实现一棵分类树(看清 Gini 计算)。
3. **过拟合**与剪枝(深度/叶子/`ccp_alpha`)。
4. 特征重要性 + 可视化。
5. 树的高方差 → 为何需要森林(5.7)。

## 目录 / TOC
1. [分裂准则: Gini 与熵 ⭐](#1)
2. [🚢 数据: Titanic](#2)
3. [从零: 一次最优分裂 ⭐](#3)
4. [sklearn 树 + 可视化](#4)
5. [过拟合与剪枝 ⭐](#5)
6. [特征重要性 + 高方差](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 分裂准则: Gini 与熵 ⭐ / Splitting Criteria

树贪心地找"哪个特征、哪个阈值"分裂能让子节点**最纯**。纯度用**不纯度**衡量, 设节点中类别比例 $p_k$:

$$\text{Gini} = 1 - \sum_k p_k^2, \qquad \text{熵 Entropy} = -\sum_k p_k\log_2 p_k$$

两者都在**纯节点(全一类)=0**, **均匀分布最大**。**信息增益** = 父节点不纯度 − 加权子节点不纯度, 树选增益最大的分裂。

**Gini vs 熵**(面试常问): 结果通常很接近; Gini 不算 log **更快**(sklearn 默认); 熵理论上更"信息论"。实务差别极小。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def gini(p): return 1 - (p**2).sum()
def entropy(p):
    p = p[p > 0]; return -(p * np.log2(p)).sum()

# 二分类下随正类比例变化 / impurity vs class balance
ps = np.linspace(0.001, 0.999, 200)
g = [gini(np.array([p, 1-p])) for p in ps]
e = [entropy(np.array([p, 1-p])) for p in ps]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, g, label="Gini (max 0.5)"); ax.plot(ps, e, label="Entropy (max 1.0)")
ax.set_xlabel("正类比例 p"); ax.set_ylabel("不纯度"); ax.legend()
ax.set_title("不纯度: 纯(p=0或1)时=0, 均匀(p=0.5)时最大")
plt.tight_layout(); plt.show()
print("两条曲线形状几乎一样 → Gini 与熵实务差别极小, sklearn 默认 Gini(更快)")


<a id="2"></a>
## 2. 数据: Titanic / The Titanic Dataset

泰坦尼克生还预测——分类教学的"另一个 Iris"。每行一名乘客, 标签 `survived`(0/1)。混合了数值(年龄/票价)和类别(舱位/性别)特征, 还有缺失值, 很贴近真实。这里先做简化清洗。


In [ ]:
df = sns.load_dataset("titanic")
print("Titanic:", df.shape, "| 生还率", f"{df['survived'].mean():.0%}")
feat = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
d = df[feat + ["survived"]].copy()
d["age"] = d["age"].fillna(d["age"].median())     # 缺失填中位数(3.2)
d["fare"] = d["fare"].fillna(d["fare"].median())
d["sex"] = (d["sex"] == "male").astype(int)        # 编码(3.5)
print(d.head(3).to_string())

from sklearn.model_selection import train_test_split
X, y = d[feat].values, d["survived"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


<a id="3"></a>
## 3. 从零: 一次最优分裂 ⭐ / Best Split From Scratch

完整树太长, 这里实现**核心**: 扫所有特征、所有候选阈值, 用 Gini 加权选最优分裂。这就是树每个节点干的事。


In [ ]:
def gini_impurity(y):
    if len(y) == 0: return 0
    p = np.bincount(y) / len(y)
    return 1 - (p**2).sum()

def best_split(X, y):
    n, d = X.shape
    parent = gini_impurity(y)
    best = {"gain": -1}
    for j in range(d):
        for thr in np.unique(X[:, j]):
            left = X[:, j] <= thr
            if left.sum() == 0 or left.sum() == n: continue
            wl, wr = left.mean(), 1 - left.mean()
            child = wl*gini_impurity(y[left]) + wr*gini_impurity(y[~left])
            gain = parent - child
            if gain > best["gain"]:
                best = {"gain": gain, "feat": j, "thr": thr}
    return best

bs = best_split(X_tr, y_tr)
print(f"根节点最优分裂: 特征 '{feat[bs['feat']]}' <= {bs['thr']}, 信息增益 {bs['gain']:.4f}")
print("(通常是 sex: 性别是 Titanic 最强预测因子 — 'women and children first')")


<a id="4"></a>
## 4. sklearn 树 + 可视化 / sklearn Tree & Visualization


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
print(f"深度3 树 test 准确率: {tree.score(X_te, y_te):.3f}")

fig, ax = plt.subplots(figsize=(15, 6))
plot_tree(tree, feature_names=feat, class_names=["died","survived"],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title("Titanic 决策树 (深度3) — 每个节点一个是/否问题")
plt.tight_layout(); plt.show()
print("可解释性是树的最大卖点: 整个决策可读成流程图")


<a id="5"></a>
## 5. 过拟合与剪枝 ⭐ / Overfitting & Pruning

不限制的树会一直分裂到每个叶子纯净——**完美拟合训练集**(训练准确率 100%)但**严重过拟合**。控制手段：
- **预剪枝**: `max_depth`、`min_samples_leaf`、`min_samples_split`。
- **后剪枝**: `ccp_alpha`(代价复杂度剪枝, 类似 4.5 Lasso 的复杂度惩罚)。


In [ ]:
depths = range(1, 21)
tr_acc, te_acc = [], []
for dep in depths:
    t = DecisionTreeClassifier(max_depth=dep, random_state=0).fit(X_tr, y_tr)
    tr_acc.append(t.score(X_tr, y_tr)); te_acc.append(t.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, tr_acc, "o-", label="训练")
ax.plot(depths, te_acc, "s-", label="测试")
ax.set_xlabel("max_depth"); ax.set_ylabel("准确率"); ax.legend()
ax.set_title("过拟合: 深度↑训练准确率→1, 测试先升后降")
plt.tight_layout(); plt.show()
print(f"满树(无限制)训练准确率: {DecisionTreeClassifier(random_state=0).fit(X_tr,y_tr).score(X_tr,y_tr):.3f} (几乎完美=过拟合)")
print(f"最佳测试深度约 {list(depths)[int(np.argmax(te_acc))]}")


In [ ]:
# 后剪枝: ccp_alpha 路径 / cost-complexity pruning
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(X_tr, y_tr)
alphas = path.ccp_alphas[:-1]
scores = [DecisionTreeClassifier(ccp_alpha=a, random_state=0).fit(X_tr,y_tr).score(X_te,y_te)
          for a in alphas]
best_a = alphas[int(np.argmax(scores))]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, scores, marker=".")
ax.axvline(best_a, color="r", ls="--", label=f"最佳 ccp_alpha={best_a:.4f}")
ax.set_xlabel("ccp_alpha (剪枝强度)"); ax.set_ylabel("test 准确率"); ax.legend()
ax.set_title("后剪枝: 增大 alpha 剪掉弱分支, 防过拟合")
plt.tight_layout(); plt.show()
print(f"最佳 ccp_alpha={best_a:.4f}, test 准确率 {max(scores):.3f}")


<a id="6"></a>
## 6. 特征重要性 + 高方差 / Importance & High Variance

- **特征重要性** = 该特征在所有分裂中带来的总不纯度下降(归一化)。比线性系数更适合非线性。
- **树是高方差模型**: 训练数据小扰动 → 完全不同的树。下面用不同随机子样本训练多棵深树, 看预测有多不稳——这正是 **5.7 随机森林用 bagging 平均掉方差** 的动机。


In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
imp = pd.Series(tree.feature_importances_, index=feat).sort_values(ascending=False)
print("特征重要性:"); print(imp.round(3).to_string())

# 高方差演示: 不同 bootstrap 上满树, 对同一批测试点预测分歧多大 / prediction instability
rng = np.random.default_rng(0)
preds = []
for _ in range(30):
    idx = rng.choice(len(X_tr), len(X_tr), replace=True)   # bootstrap
    t = DecisionTreeClassifier(random_state=0).fit(X_tr[idx], y_tr[idx])
    preds.append(t.predict(X_te))
preds = np.array(preds)                       # (30 棵树, n_test)
# 每个测试点被预测为 1 的比例, 越靠近 0.5 越不稳
frac1 = preds.mean(0)
unstable = ((frac1 > 0.2) & (frac1 < 0.8)).mean()
print(f"\n30 棵 bootstrap 满树: {unstable:.0%} 的测试样本预测在树间摇摆(被预测为1的比例在20%~80%)")
print(f"单棵满树 test 准确率波动: {preds.mean(1).std():.3f} 量级的标准差")
print("→ 满树对数据扰动敏感(高方差); 5.7 用 bagging 平均多棵树降方差")


<a id="7"></a>
## 7. 小结 / Summary

```
决策树: 贪心选分裂, 最大化信息增益(父-加权子不纯度); Gini/熵实务等价(Gini 更快)
轴对齐方块边界; 可解释性强; 不需缩放; 处理混合类型
过拟合: 满树训练准确率→100%; 预剪枝(max_depth/min_samples) + 后剪枝(ccp_alpha)
特征重要性 = 总不纯度下降; 树是高方差模型 → 引出 bagging(5.7)
```

### 💡 面试速查
1. **Gini=1-Σp², 熵=-Σp·log p**; 信息增益选分裂; Gini 更快(默认)
2. **防过拟合**: max_depth / min_samples_leaf / ccp_alpha
3. **不需缩放** —— 分裂只比较阈值, 与量纲无关
4. **高方差** —— 数据小扰动→不同树 → 森林(bagging)平均降方差
5. **特征重要性**基于不纯度下降, 注意对高基数特征有偏

### 下一节
**5.7 随机森林**——把很多高方差的树 bagging + 特征随机, 平均掉方差, 几乎不需调参的强力开箱即用模型。
